In [ ]:
# random forest , decision tree, gradient boost classifier

In [ ]:
#The training dataset, intended solely for training purposes, has been imported, 
#featuring MACCS keys that represent the structural characteristics of the chemical compounds, 
#with each chemical labeled as 0 for non-developmental toxicants and 1 for developmental toxicants.

In [6]:
import pandas as pd 
df1=pd.read_excel('dev_tox.xlsx')
df1=df1.iloc[:, 1:]

In [7]:
import numpy as np 
import sklearn as sk
from sklearn import metrics
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import validation_curve
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV

from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

In [8]:
X3=df1.drop(['label'], axis=1)
y=df1['label']
X_trainval, X_test, y_trainval, y_test=train_test_split(X3, y, random_state=42)
X_train, X_valid, y_train, y_valid=train_test_split(X_trainval, y_trainval, random_state=1)

In [9]:
X_train.columns

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166],
      dtype='object', length=167)

In [ ]:
# Construt three tree models 

In [10]:
import itertools
import math
import random
from typing import Callable, List, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [11]:
def multi_tree_model(random_state = 42): 

    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.ensemble import GradientBoostingClassifier


    
    decision_tree_model=DecisionTreeClassifier(random_state = 42)
    random_forest_model=RandomForestClassifier(n_jobs=-1, random_state=42)
    gradient_boost_model = GradientBoostingClassifier(random_state=42)
    
    decision_tree_model.fit(X_train, y_train)
    random_forest_model.fit(X_train, y_train)
    gradient_boost_model.fit(X_train, y_train)
    
    decision_tree_score=decision_tree_model.score(X_test, y_test)
    random_forest_score=random_forest_model.score(X_test, y_test)
    gradient_boost_score=gradient_boost_model.score(X_test, y_test)


    return (decision_tree_score,random_forest_score, gradient_boost_score )

In [12]:
multi_tree_model()

(0.7253694581280788, 0.8103448275862069, 0.7586206896551724)

In [8]:
#construct Xgboost and optimize the hyperparameters 

In [15]:
import numpy as np
import xgboost as xgb
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier




model = GradientBoostingClassifier()


param_grid = {
    'n_estimators': [20, 50, 100, 150],
    'max_depth': [3, 5, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.5, 0.8, 1.0],

}


grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=5)


grid_search.fit(X_trainval, y_trainval)


best_params = grid_search.best_params_
best_score = grid_search.best_score_


print("Best Hyperparameters:", best_params)
print("Best Cross-Validation Score:", best_score)


best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print("Test Set Accuracy:", test_score)

Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 150, 'subsample': 0.5}
Best Cross-Validation Score: 0.7967145790554414
Test Set Accuracy: 0.7967980295566502


In [ ]:
# train model with best hyperparameters 

In [21]:
final_model = GradientBoostingClassifier(
    
    **best_params
)


In [22]:
final_model.fit(X_trainval, y_trainval)

GradientBoostingClassifier(max_depth=7, n_estimators=150, subsample=0.5)

In [23]:
from sklearn.metrics import accuracy_score, classification_report ,log_loss
y_pred = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)[:, 1]  # Get probabilities for the positive class


accuracy = accuracy_score(y_test, y_pred)
cross_entropy_loss = log_loss(y_test, y_pred_prob)

print(f"Test Set Accuracy: {accuracy:.4f}")
print(f"Cross-Entropy Loss (Log Loss): {cross_entropy_loss:.4f}")

Test Set Accuracy: 0.8030
Cross-Entropy Loss (Log Loss): 0.4636


In [ ]:
# Feature seletion 

In [ ]:
#construct a new Xggoost classifier and leverage feature importance to select imporant features 

In [80]:
best_params = grid_search.best_params_

In [24]:
best_params

{'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 150, 'subsample': 0.5}

In [ ]:
# leverage gradient boost classier feature importance function to do feature selection 

In [49]:
def GradientBoostingClassifier(test_size: Union[float, int] = 0.2, random_state: int = 42):
    
    model, features, score = (None,) * 3
    
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.model_selection import cross_val_score
    

    
    features=X_train.columns
    print(features)
    features_1=features[0:50]
    features_2=features[50:200]
    features_3=features[100:166]

    
    feature_seg=[features_1,features_2,features_3]
    
 
    
    final_features=[]
    
    for feat in feature_seg:
        
        X_train_fe=X_train[feat]
        X_test_fe=X_test[feat]
    
        clf = GradientBoostingClassifier(**best_params)
        clf.fit(X_train_fe, y_train)

        
              
        feature_importances = clf.feature_importances_ 

    
        feature_importances_df = pd.DataFrame({
        'features': X_train_fe.columns,
        'importance': feature_importances
        })
        
        feature_importances_df = feature_importances_df.sort_values(by='importance', ascending=False)
        final_features.extend(feature_importances_df['features'][0:20].values)


#    print('this is final feature', final_features)
    
    X_train_final=X_train[final_features]

    X_test_final=X_test[final_features]
    
    clf_final  = GradientBoostingClassifier(**best_params)

    clf_final.fit(X_train_final, y_train)

    feature_importances_final = clf_final.feature_importances_                                                            
        
    feature_importances_df_final = pd.DataFrame({
        'features': X_train_final.columns,
        'importance': feature_importances_final
        })
        
    feature_importances_df_final = feature_importances_df_final.sort_values(by='importance', ascending=False)
    top_30_features=feature_importances_df_final['features'][0:50]
                                                
    y_pred_proba_final = clf_final.predict_proba(X_test_final)[:,1]
    
    roc_auc_result = roc_auc_score(y_test, y_pred_proba_final)

    

    X_train_result=X_train[top_30_features.values]
    X_test_result=X_test[top_30_features.values]
    
    clf_result =   GradientBoostingClassifier(**best_params)
    clf_result.fit(X_train_result, y_train)
    
    y_pred_proba_result = clf_result.predict_proba(X_test_result)[:,1]
        

    roc_auc_result = roc_auc_score(y_test, y_pred_proba_result)
    
    top_30_features=np.array(top_30_features)


    return (clf_result, top_30_features,roc_auc_result)

In [50]:
GradientBoostingClassifier(0.2,42)

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166],
      dtype='object', length=167)


(GradientBoostingClassifier(max_depth=7, n_estimators=150, subsample=0.5),
 array([138, 134, 93, 154, 161, 163, 72, 149, 159, 48, 152, 166, 126, 165,
        141, 114, 35, 115, 132, 49, 131, 161, 129, 139, 112, 10, 112, 162,
        132, 108, 3, 131, 129, 134, 140, 139, 140, 20, 114, 162, 136, 34,
        146, 29, 7, 108, 146, 136, 28, 115], dtype=object),
 0.8231169557536611)